# MolMage on Colab

Turn a molecule (SMILES or SDF) into a **MolMage**, the 64 × 64 × 4 surface image used by [ImagenMol](https://anonymous.4open.science/r/ImagenMol-65A8/README.md), decode it back into a 3D mesh, and look at Blender snapshots of that mesh.

`SMILES / SDF → RDKit conformer → PyMOL surface → Open3D UV atlas → Blender bake → MolMage → mesh + snapshots`

Run the cells in order; a CPU runtime is enough. The code lives in the [`molmage_colab`](https://github.com/Qian-Anton/molmage-colab) package, which step 1 clones, so each cell only sets a few options. Results are saved in `/content/molmage_outputs/<name>/` (MolMage `.npy`, mesh `.glb`, snapshots).

In [ ]:
#@title 1 · Setup (first run ≈ 3–5 min) { display-mode: "form" }
#@markdown Clones the helper package and builds a separate Python 3.10 environment with bpy 4.0.0, PyMOL, Open3D and RDKit. Colab's own Python is left untouched.
REPO_URL = "https://github.com/Qian-Anton/molmage-colab.git"  #@param {type:"string"}
FORCE_REINSTALL = False  #@param {type:"boolean"}

import os, shutil, subprocess, sys

env = dict(os.environ, GIT_TERMINAL_PROMPT="0")  # fail instead of asking for a password
repo = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if os.path.isdir(repo) and set(os.listdir(repo)) <= {".git"}:
    shutil.rmtree(repo)  # leftover of a failed or interrupted clone
if os.path.isdir(repo):
    subprocess.run(["git", "-C", repo, "pull", "-q", "--ff-only"], env=env, capture_output=True)
else:
    clone = subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, repo], env=env,
                           capture_output=True, text=True)
    if clone.returncode:
        raise RuntimeError(f"git clone failed; check that {REPO_URL} exists and is public.\n{clone.stderr.strip()}")
if repo not in sys.path:
    sys.path.insert(0, repo)
for module in [m for m in sys.modules if m.split(".")[0] == "molmage_colab"]:
    del sys.modules[module]  # use the freshly pulled code

import molmage_colab as mc
versions = mc.setup(force=FORCE_REINSTALL)

In [ ]:
#@title 2 · Molecule → MolMage { display-mode: "form" }
INPUT = "SMILES"  #@param ["SMILES", "SDF file", "MolMage file (.npy / .pt)"]
SMILES = "CC(=O)Oc1ccccc1C(=O)O"  #@param {type:"string"}
#@markdown `FILE` (SDF or MolMage input): `upload`, an example SDF, or a path such as `/content/my.sdf`. `INDEX` picks the record in a multi-molecule SDF or the sample in a batched `.pt`.
FILE = "upload"  #@param ["upload", "aspirin.sdf", "caffeine.sdf", "ibuprofen.sdf", "lidocaine.sdf", "acetaminophen.sdf"] {allow-input: true}
INDEX = 0  #@param {type:"integer"}
#@markdown ---
RESOLUTION = 64  #@param [64, 128] {type:"raw"}
SEED = 42  #@param {type:"integer"}
SKIP_FILTERS = False  #@param {type:"boolean"}

result = mc.run(INPUT, smiles=SMILES, file=FILE, index=INDEX,
                resolution=RESOLUTION, seed=SEED, skip_filters=SKIP_FILTERS)

In [ ]:
#@title 3 · Mesh snapshots (Blender) { display-mode: "form" }
#@markdown Renders the mesh decoded from the MolMage with the official preview settings; 0° is the view used in `_mmg.png`.
VIEWS = 4  #@param {type:"slider", min:1, max:8, step:1}

mc.show_snapshots(views=VIEWS)

In [ ]:
#@title 4 · Interactive 3D view { display-mode: "form" }
#@markdown Left: the conformer inside the PyMOL surface that was encoded. Right: the mesh decoded from the MolMage. `position` paints both with the MolMage x, y, z colours.
COLOR_BY = "patch"  #@param ["patch", "position"]

mc.show_mesh(color_by=COLOR_BY)